# 3.8 Temporal Consistency: Cosine Similarity Analysis

This notebook analyzes the temporal consistency of latent representations across multiple scans per patient.

## Research Questions:
1. How similar are a patient's latent representations across different timepoints?
2. Does the autoencoder produce stable, reproducible representations?
3. Are changes in latent space correlated with disease progression?
4. Which latent dimensions are most stable vs. most variable?

## Methods:
- Calculate cosine similarity between all pairs of scans for each patient
- Analyze similarity by time interval (baseline vs follow-up)
- Correlate similarity changes with SBR changes
- Identify stable vs variable latent dimensions

## Section 1: Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
from scipy import stats
from scipy.spatial.distance import cosine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

print("Libraries imported successfully!")

### 1.1 Define Constants

In [ ]:
# Feature columns
FEATURE_COLS = [f'latent_{i}' for i in range(256)]
TARGETS = ['SBR_PC1', 'SBR_PC2', 'SBR_PC3']
SBR_COLS = [
    'DATSCAN_CAUDATE_R', 'DATSCAN_CAUDATE_L',
    'DATSCAN_PUTAMEN_R', 'DATSCAN_PUTAMEN_L',
    'DATSCAN_PUTAMEN_R_ANT', 'DATSCAN_PUTAMEN_L_ANT'
]

print(f"Features: {len(FEATURE_COLS)}")
print(f"Targets: {TARGETS}")

### 1.2 Load Data

In [ ]:
# Load merged clinical data
df_merged = pd.read_csv('output/merged_data.csv')

# Load latent vectors
latent_file = 'output/Experiments/LatentVectorAnalysis/train_latent_vectors_with_patno.csv'
df_latent = pd.read_csv(latent_file)

print(f"Merged clinical data: {df_merged.shape}")
print(f"Latent vectors: {df_latent.shape}")

# Merge datasets
df_latent_clean = df_latent.dropna(subset=['PATNO']).copy()
df_combined = pd.merge(df_latent_clean, df_merged, on='PATNO', how='inner')

print(f"Combined dataset: {df_combined.shape}")
print(f"Unique patients: {df_combined['PATNO'].nunique()}")

### 1.3 Calculate SBR PCs

In [ ]:
# Calculate SBR PCs
df_with_sbr = df_combined.dropna(subset=SBR_COLS).copy()
scaler_sbr = StandardScaler()
sbr_scaled = scaler_sbr.fit_transform(df_with_sbr[SBR_COLS])
pca = PCA(n_components=3, random_state=0)
sbr_pcs = pca.fit_transform(sbr_scaled)
df_with_sbr['SBR_PC1'] = sbr_pcs[:, 0]
df_with_sbr['SBR_PC2'] = sbr_pcs[:, 1]
df_with_sbr['SBR_PC3'] = sbr_pcs[:, 2]
df_combined = df_with_sbr.copy()

print(f"Dataset with SBR PCs: {df_combined.shape}")

## Section 2: Cosine Similarity Analysis

### 2.1 Calculate Pairwise Cosine Similarities

In [ ]:
print("\n" + "="*80)
print("COSINE SIMILARITY ANALYSIS")
print("="*80)

# Group by patient
patient_groups = df_combined.groupby('PATNO')

# Calculate cosine similarities
similarity_results = []

for patno, group in patient_groups:
    if len(group) < 2:
        continue  # Need at least 2 scans
    
    # Extract latent vectors
    latent_vecs = group[FEATURE_COLS].values
    
    # Calculate all pairwise cosine similarities
    n_scans = len(latent_vecs)
    similarities = []
    
    for i in range(n_scans):
        for j in range(i+1, n_scans):
            # Cosine similarity = 1 - cosine distance
            cos_sim = 1 - cosine(latent_vecs[i], latent_vecs[j])
            similarities.append(cos_sim)
    
    if len(similarities) > 0:
        # Get SBR values
        sbr_pc1_vals = group['SBR_PC1'].values
        sbr_pc2_vals = group['SBR_PC2'].values
        sbr_pc3_vals = group['SBR_PC3'].values
        
        # Calculate SBR changes
        sbr_pc1_change = np.max(sbr_pc1_vals) - np.min(sbr_pc1_vals)
        sbr_pc2_change = np.max(sbr_pc2_vals) - np.min(sbr_pc2_vals)
        sbr_pc3_change = np.max(sbr_pc3_vals) - np.min(sbr_pc3_vals)
        
        similarity_results.append({
            'PATNO': patno,
            'N_Scans': n_scans,
            'N_Pairs': len(similarities),
            'Mean_Cosine_Similarity': np.mean(similarities),
            'Std_Cosine_Similarity': np.std(similarities),
            'Min_Cosine_Similarity': np.min(similarities),
            'Max_Cosine_Similarity': np.max(similarities),
            'SBR_PC1_Change': sbr_pc1_change,
            'SBR_PC2_Change': sbr_pc2_change,
            'SBR_PC3_Change': sbr_pc3_change,
            'Mean_SBR_PC1': np.mean(sbr_pc1_vals),
            'Mean_SBR_PC2': np.mean(sbr_pc2_vals),
            'Mean_SBR_PC3': np.mean(sbr_pc3_vals)
        })

df_similarity = pd.DataFrame(similarity_results)

print(f"\nPatients with multiple scans: {len(df_similarity)}")
print(f"Total scan pairs analyzed: {df_similarity['N_Pairs'].sum()}")
print(f"\nCosine Similarity Statistics:")
print(f"  Mean: {df_similarity['Mean_Cosine_Similarity'].mean():.4f}")
print(f"  Std:  {df_similarity['Mean_Cosine_Similarity'].std():.4f}")
print(f"  Min:  {df_similarity['Mean_Cosine_Similarity'].min():.4f}")
print(f"  Max:  {df_similarity['Mean_Cosine_Similarity'].max():.4f}")

### 2.2 Summary Statistics

In [ ]:
print("\n" + "="*80)
print("TEMPORAL CONSISTENCY SUMMARY")
print("="*80)

print(f"\nScan Distribution:")
print(df_similarity['N_Scans'].value_counts().sort_index().to_string())

print(f"\nCosine Similarity by Number of Scans:")
for n_scans in sorted(df_similarity['N_Scans'].unique()):
    subset = df_similarity[df_similarity['N_Scans'] == n_scans]
    print(f"  {n_scans} scans (n={len(subset)}): mean={subset['Mean_Cosine_Similarity'].mean():.4f}, std={subset['Mean_Cosine_Similarity'].std():.4f}")

print(f"\nTop 10 Most Similar Patients (Highest Cosine Similarity):")
print(df_similarity.nlargest(10, 'Mean_Cosine_Similarity')[['PATNO', 'N_Scans', 'Mean_Cosine_Similarity', 'SBR_PC1_Change']].to_string(index=False))

print(f"\nTop 10 Most Variable Patients (Lowest Cosine Similarity):")
print(df_similarity.nsmallest(10, 'Mean_Cosine_Similarity')[['PATNO', 'N_Scans', 'Mean_Cosine_Similarity', 'SBR_PC1_Change']].to_string(index=False))

### 2.3 Correlation with SBR Changes

In [ ]:
print("\n" + "="*80)
print("CORRELATION: COSINE SIMILARITY vs SBR CHANGES")
print("="*80)

# Correlate cosine similarity with SBR changes
for target in ['SBR_PC1_Change', 'SBR_PC2_Change', 'SBR_PC3_Change']:
    # Remove NaN values
    valid_data = df_similarity.dropna(subset=['Mean_Cosine_Similarity', target])
    
    if len(valid_data) > 2:
        corr, p_val = stats.pearsonr(valid_data['Mean_Cosine_Similarity'], valid_data[target])
        print(f"\n{target}:")
        print(f"  Pearson r: {corr:.4f}")
        print(f"  p-value: {p_val:.4e}")
        
        if p_val < 0.05:
            print(f"  → Significant correlation (p<0.05)")
        else:
            print(f"  → No significant correlation (p≥0.05)")

## Section 3: Latent Dimension Stability

### 3.1 Identify Stable vs Variable Dimensions

In [ ]:
print("\n" + "="*80)
print("LATENT DIMENSION STABILITY ANALYSIS")
print("="*80)

# For each latent dimension, calculate variance across scans per patient
dimension_stability = []

for dim_idx in range(len(FEATURE_COLS)):
    dim_name = FEATURE_COLS[dim_idx]
    
    # Calculate coefficient of variation for each patient
    cv_values = []
    
    for patno, group in patient_groups:
        if len(group) < 2:
            continue
        
        dim_values = group[dim_name].values
        mean_val = np.mean(dim_values)
        std_val = np.std(dim_values)
        
        if mean_val != 0:
            cv = std_val / np.abs(mean_val)
            cv_values.append(cv)
    
    if len(cv_values) > 0:
        dimension_stability.append({
            'Dimension': dim_name,
            'Mean_CV': np.mean(cv_values),
            'Std_CV': np.std(cv_values),
            'Median_CV': np.median(cv_values)
        })

df_stability = pd.DataFrame(dimension_stability)
df_stability = df_stability.sort_values('Mean_CV')

print(f"\nMost Stable Dimensions (Low Coefficient of Variation):")
print(df_stability.head(10)[['Dimension', 'Mean_CV', 'Std_CV']].to_string(index=False))

print(f"\nMost Variable Dimensions (High Coefficient of Variation):")
print(df_stability.tail(10)[['Dimension', 'Mean_CV', 'Std_CV']].to_string(index=False))

## Section 4: Visualizations

### 4.1 Cosine Similarity Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Histogram of mean cosine similarities
axes[0, 0].hist(df_similarity['Mean_Cosine_Similarity'], bins=30, color='steelblue', alpha=0.8, edgecolor='black')
axes[0, 0].axvline(df_similarity['Mean_Cosine_Similarity'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df_similarity['Mean_Cosine_Similarity'].mean():.3f}")
axes[0, 0].set_xlabel('Mean Cosine Similarity', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Distribution of Cosine Similarities Across Patients', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Cosine similarity by number of scans
for n_scans in sorted(df_similarity['N_Scans'].unique()):
    subset = df_similarity[df_similarity['N_Scans'] == n_scans]
    axes[0, 1].scatter([n_scans]*len(subset), subset['Mean_Cosine_Similarity'], alpha=0.6, s=100, label=f"n={len(subset)}")

axes[0, 1].set_xlabel('Number of Scans per Patient', fontsize=11)
axes[0, 1].set_ylabel('Mean Cosine Similarity', fontsize=11)
axes[0, 1].set_title('Cosine Similarity vs Number of Scans', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Cosine similarity vs SBR_PC1 change
axes[1, 0].scatter(df_similarity['SBR_PC1_Change'], df_similarity['Mean_Cosine_Similarity'], alpha=0.6, s=100, color='coral', edgecolor='black')
axes[1, 0].set_xlabel('SBR_PC1 Change', fontsize=11)
axes[1, 0].set_ylabel('Mean Cosine Similarity', fontsize=11)
axes[1, 0].set_title('Cosine Similarity vs SBR_PC1 Change', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Add correlation
valid_data = df_similarity.dropna(subset=['Mean_Cosine_Similarity', 'SBR_PC1_Change'])
if len(valid_data) > 2:
    corr, p_val = stats.pearsonr(valid_data['Mean_Cosine_Similarity'], valid_data['SBR_PC1_Change'])
    axes[1, 0].text(0.05, 0.95, f'r={corr:.3f}, p={p_val:.3e}', transform=axes[1, 0].transAxes, 
                    fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Dimension stability
top_stable = df_stability.head(15)
axes[1, 1].barh(range(len(top_stable)), top_stable['Mean_CV'].values, color='steelblue', alpha=0.8, edgecolor='black')
axes[1, 1].set_yticks(range(len(top_stable)))
axes[1, 1].set_yticklabels(top_stable['Dimension'].values, fontsize=9)
axes[1, 1].set_xlabel('Mean Coefficient of Variation', fontsize=11)
axes[1, 1].set_title('Top 15 Most Stable Latent Dimensions', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('output/cosine_similarity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("Plot saved to: output/cosine_similarity_analysis.png")

## Section 5: Save Results

In [ ]:
output_dir = Path('output')

# Save similarity results
df_similarity.to_csv(output_dir / 'cosine_similarity_by_patient.csv', index=False)

# Save dimension stability
df_stability.to_csv(output_dir / 'latent_dimension_stability.csv', index=False)

print("Results saved to:")
print(f"  - {output_dir / 'cosine_similarity_by_patient.csv'}")
print(f"  - {output_dir / 'latent_dimension_stability.csv'}")

print(f"\nSimilarity Summary:")
print(df_similarity[['PATNO', 'N_Scans', 'Mean_Cosine_Similarity', 'SBR_PC1_Change']].head(20).to_string(index=False))

## Interpretation Guide

### Cosine Similarity Interpretation:

**High Cosine Similarity (0.9-1.0):**
- ✅ Stable latent representations across scans
- ✅ Autoencoder produces reproducible features
- ✅ Minimal disease progression or scanner variation

**Moderate Cosine Similarity (0.7-0.9):**
- ⚠️ Some variation in latent space
- ⚠️ Could indicate disease progression or scan-to-scan variability

**Low Cosine Similarity (<0.7):**
- ❌ Large changes in latent representations
- ❌ Possible disease progression or significant pathology changes

### Dimension Stability:

**Stable Dimensions (Low CV):**
- Represent consistent biological features
- Good for longitudinal tracking
- Less affected by scanner/acquisition differences

**Variable Dimensions (High CV):**
- Capture disease-specific changes
- Sensitive to disease progression
- May be affected by scanner differences

### Clinical Implications:

- **Reproducibility**: High cosine similarity validates autoencoder stability
- **Longitudinal tracking**: Stable dimensions enable disease monitoring
- **Disease progression**: Low similarity correlates with pathology changes
- **Quality control**: Identifies patients with inconsistent scans